# 深層学習DIA代表パターン選択

前回（[#15b 深層学習DIA大規模階層クラスタリング](../blog/article-15b-openms-stage-clustering.md)）では、15,000+の有意タンパク質を50クラスターに分割しました。この記事では、この50クラスターから**代表的な8パターン**を系統的に選択し、大腸がんステージ進行に最も重要な分子動態パターンを特定します。

**🎯 パターン選択の特徴:**
- **入力**: 50クラスターの階層クラスタリング結果
- **選択基準**: クラスターサイズ・ステージ間変動性・パターン間相関の統合評価
- **出力**: 代表的な8パターンで大腸がんステージ進行を包括的に表現
- **アルゴリズム**: 多基準評価 + 貪欲法による多様性最大化

**対応記事**: [#15c 深層学習DIA代表パターン選択](../blog/article-15c-openms-stage-pattern-selection.md)

## ライブラリと設定

In [ ]:
import numpy as np             # 数値計算ライブラリ: クラスター統計計算とパターン分析に使用
import pandas as pd            # データ分析ライブラリ: クラスター結果とステージ情報の統合処理に使用
import matplotlib.pyplot as plt  # グラフ描画ライブラリ: パターン選択結果の可視化に使用
from scipy.stats import pearsonr  # パターン間相関分析用
import seaborn as sns          # ヒートマップ描画用

# --- 深層学習DIA解析用の定数設定 ---
RESULTS = "../results"         # 解析結果の出力先ディレクトリパス
FIG_DIR = f"{RESULTS}/figures"   # 図の保存先ディレクトリパス
TABLE_DIR = f"{RESULTS}/tables"  # テーブル（CSV/Excel）の保存先ディレクトリパス

# ステージの表示順序を定義（論文と同じNormal→Stage I→II→III→IVの順）
STAGE_ORDER = ["Normal", "I", "II", "III", "IV"]

# パターン選択用評価パラメータ
MIN_CLUSTER_SIZE = 10          # クラスター最小サイズ（統計的信頼性確保）
VARIATION_THRESHOLD = 0.3      # 変動性の最小閾値（上位70%の変動性を要求）
N_REPRESENTATIVE_PATTERNS = 8  # 選択する代表パターン数

print(f"深層学習DIA パターン選択設定:")
print(f"  最小クラスターサイズ: {MIN_CLUSTER_SIZE} タンパク質")
print(f"  変動性閾値: 上位{(1-VARIATION_THRESHOLD)*100:.0f}%")
print(f"  選択パターン数: {N_REPRESENTATIVE_PATTERNS}")
print(f"  出力先: {TABLE_DIR}")

## データ読み込みとクラスター統計計算

前回の階層クラスタリング結果を読み込み、各クラスターの基本統計（サイズ、変動性、パターン形状）を計算します。

In [ ]:
# --- 前処理済みデータとクラスタリング結果の統合 ---
# 深層学習DIA解析結果（19,981タンパク質）
df = pd.read_csv(f"{RESULTS}/preprocessed_data_openms.csv", index_col=0)
sample_info = pd.read_csv(f"{RESULTS}/sample_info_openms.csv")

# 前回のクラスター割り当て結果
cluster_df = pd.read_csv(f"{TABLE_DIR}/cluster_assignments_openms.csv")

print(f"データ統合完了:")
print(f"- 深層学習DIA: {df.shape[0]} タンパク質 × {df.shape[1]} サンプル")
print(f"- クラスター割り当て: {len(cluster_df)} タンパク質、{cluster_df['Cluster'].nunique()} クラスター")

# ステージ別中央値の再計算
print("\nステージ別中央値の再計算...")
stages = [s for s in STAGE_ORDER if s in sample_info["Stage"].values]
medians = {}

for s in stages:
    cols = [c for c in sample_info[sample_info["Stage"] == s]["Sample"]
            if c in df.columns]
    if cols:
        medians[s] = df[cols].median(axis=1)
        print(f"  Stage {s}: {len(cols)} サンプル")

median_df = pd.DataFrame(medians)
print(f"ステージ別中央値マトリクス: {median_df.shape[0]} タンパク質 × {median_df.shape[1]} ステージ")

In [ ]:
# --- クラスター別統計の計算 ---
def calculate_cluster_statistics(cluster_df, median_df):
    """各クラスターの基本統計を計算: サイズ・変動性・パターン形状。"""
    cluster_stats = []

    print("クラスター別統計計算中...")
    for cluster_id in sorted(cluster_df["Cluster"].unique()):
        # 該当クラスターのタンパク質を取得
        proteins = cluster_df[cluster_df["Cluster"] == cluster_id]["Protein"].tolist()

        # ステージ別中央値データを取得
        cluster_medians = median_df.loc[median_df.index.isin(proteins)]
        if cluster_medians.empty:
            print(f"  Cluster {cluster_id}: データなし (スキップ)")
            continue

        # Z-score正規化（行方向 = ステージ間）
        cluster_zscore = cluster_medians.apply(
            lambda x: (x - x.mean()) / x.std() if x.std() > 0 else x * 0, axis=1
        ).dropna()

        if cluster_zscore.empty:
            print(f"  Cluster {cluster_id}: 正規化後データなし (スキップ)")
            continue

        # クラスター平均パターンを計算
        pattern = cluster_zscore.mean(axis=0)
        # ステージ間変動の明瞭性（標準偏差）を計算
        pattern_std = pattern.std()
        # 最大変動幅（min-max差）を計算
        pattern_range = pattern.max() - pattern.min()

        cluster_stats.append({
            "Cluster": cluster_id,
            "Size": len(cluster_zscore),
            "Pattern_Std": pattern_std,
            "Pattern_Range": pattern_range,
            "Pattern": pattern.tolist(),
            "Normal_Value": pattern.iloc[0],  # Normal時の値
            "StageIV_Value": pattern.iloc[-1]  # Stage IV時の値
        })
        
        if cluster_id % 10 == 0:
            print(f"  進捗: Cluster {cluster_id} 完了")

    return pd.DataFrame(cluster_stats)

# クラスター統計の計算実行
cluster_stats_df = calculate_cluster_statistics(cluster_df, median_df)

print(f"\nクラスター統計計算完了: {len(cluster_stats_df)} クラスター")
print(f"サイズ分布: {cluster_stats_df['Size'].min()}-{cluster_stats_df['Size'].max()} タンパク質")
print(f"変動性分布: {cluster_stats_df['Pattern_Std'].min():.3f}-{cluster_stats_df['Pattern_Std'].max():.3f}")

# 統計サマリーの表示
print(f"\n=== クラスター統計サマリー ===")
print(f"平均クラスターサイズ: {cluster_stats_df['Size'].mean():.1f} ± {cluster_stats_df['Size'].std():.1f}")
print(f"平均変動性: {cluster_stats_df['Pattern_Std'].mean():.3f} ± {cluster_stats_df['Pattern_Std'].std():.3f}")
print(f"平均変動幅: {cluster_stats_df['Pattern_Range'].mean():.3f} ± {cluster_stats_df['Pattern_Range'].std():.3f}")

## 代表パターン選択アルゴリズム

多基準評価による代表パターン選択を実行します。選択基準は以下の優先順位で適用されます：

1. **クラスター内タンパク質数**（最低10個以上：統計的信頼性）
2. **ステージ間変動の明瞭性**（上位70%以上：生物学的意義）
3. **パターンの多様性**（相関の低い組み合わせ：包括性確保）

In [ ]:
def select_representative_patterns(cluster_stats_df,
                                 min_size=MIN_CLUSTER_SIZE,
                                 variation_threshold=VARIATION_THRESHOLD,
                                 n_patterns=N_REPRESENTATIVE_PATTERNS):
    """多基準評価による代表パターン選択: 深層学習DIA用最適化版。

    【選択基準の優先順位】
      1. クラスター内タンパク質数（最低10個以上: 統計的信頼性）
      2. ステージ間変動の明瞭性（上位70%以上: 生物学的意義）
      3. パターンの多様性（相関の低い組み合わせ: 包括性確保）
    """
    # ステップ1: サイズ・変動性による候補絞り込み
    candidates = cluster_stats_df[
        (cluster_stats_df["Size"] >= min_size) &  # 最低10個のタンパク質
        (cluster_stats_df["Pattern_Std"] >= cluster_stats_df["Pattern_Std"].quantile(variation_threshold))
    ].copy()

    # 変動性でソート（最大変動を優先）
    candidates = candidates.sort_values("Pattern_Std", ascending=False)

    print(f"=== パターン選択プロセス ===")
    print(f"選択基準による絞り込み結果:")
    print(f"  サイズ >= {min_size}: {len(cluster_stats_df[cluster_stats_df['Size'] >= min_size])} / {len(cluster_stats_df)} クラスター")
    print(f"  変動性 >= {variation_threshold:.1%}: {len(candidates)} / {len(cluster_stats_df)} クラスター")

    # ステップ2: 多様性を考慮したパターン選択
    if len(candidates) <= n_patterns:
        # 候補が目標数以下なら全選択
        selected_clusters = candidates["Cluster"].tolist()
        print(f"  候補数が目標以下のため全選択: {len(selected_clusters)} パターン")
    else:
        # 貪欲法で最も多様なパターンセットを構築
        selected_clusters = [candidates.iloc[0]["Cluster"]]  # 最大変動パターンから開始

        print(f"  多様性選択アルゴリズム開始...")
        print(f"    初期選択: Cluster {selected_clusters[0]} (最大変動: {candidates.iloc[0]['Pattern_Std']:.3f})")

        # 残りのパターンを順次選択
        iteration = 1
        while len(selected_clusters) < n_patterns and len(selected_clusters) < len(candidates):
            remaining = candidates[~candidates["Cluster"].isin(selected_clusters)]
            if remaining.empty:
                break

            best_candidate = None
            min_max_corr = float("inf")

            # 既選択パターンとの最大相関が最小となる候補を選択
            for _, candidate in remaining.iterrows():
                max_corr = 0
                for selected_id in selected_clusters:
                    selected_pattern = candidates[candidates["Cluster"] == selected_id]["Pattern"].iloc[0]
                    # ピアソン相関係数を計算
                    corr = abs(np.corrcoef(candidate["Pattern"], selected_pattern)[0, 1])
                    max_corr = max(max_corr, corr)

                # 最大相関が最小の候補を選択
                if max_corr < min_max_corr:
                    min_max_corr = max_corr
                    best_candidate = candidate["Cluster"]

            if best_candidate is not None:
                selected_clusters.append(best_candidate)
                print(f"    第{iteration+1}選択: Cluster {best_candidate} (最大相関: {min_max_corr:.3f})")
                iteration += 1
            else:
                print(f"    選択終了: 適切な候補が見つからない")
                break

    print(f"\n代表パターン選択完了: {len(selected_clusters)} パターン")
    return selected_clusters

# 代表パターンの選択実行
representative_clusters = select_representative_patterns(cluster_stats_df)

# 選択結果の詳細表示
print(f"\n=== 選択された代表パターンの詳細 ===")
selected_stats = cluster_stats_df[cluster_stats_df["Cluster"].isin(representative_clusters)]
selected_stats_sorted = selected_stats.sort_values("Pattern_Std", ascending=False)

for i, (_, row) in enumerate(selected_stats_sorted.iterrows()):
    normal_val = row["Normal_Value"]
    stage4_val = row["StageIV_Value"]
    change_direction = "↑" if stage4_val > normal_val else "↓"
    change_magnitude = abs(stage4_val - normal_val)
    
    print(f"Pattern {i+1}: Cluster {row['Cluster']}")
    print(f"  - サイズ: {row['Size']} タンパク質")
    print(f"  - 変動性: {row['Pattern_Std']:.3f}")
    print(f"  - 変化: Normal({normal_val:.2f}) → Stage IV({stage4_val:.2f}) {change_direction} (幅: {change_magnitude:.2f})")
    print()

## パターン選択結果の可視化

選択されたパターンの特徴と選択基準の効果を可視化します。

In [ ]:
def visualize_pattern_selection_result(cluster_stats_df, representative_clusters):
    """代表パターン選択結果の可視化: 選択基準と選択されたパターンの概要表示。"""
    # フィギュアの設定（2行2列のサブプロット）
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

    # 1. クラスターサイズ分布（選択vs非選択）
    selected_mask = cluster_stats_df["Cluster"].isin(representative_clusters)

    ax1.hist(cluster_stats_df[~selected_mask]["Size"], bins=20, alpha=0.7,
             label=f"非選択 ({(~selected_mask).sum()})", color="lightgray")
    ax1.hist(cluster_stats_df[selected_mask]["Size"], bins=20, alpha=0.8,
             label=f"選択 ({selected_mask.sum()})", color="red")
    ax1.axvline(MIN_CLUSTER_SIZE, color="blue", linestyle="--",
                label=f"最小サイズ閾値 ({MIN_CLUSTER_SIZE})")
    ax1.set_xlabel("クラスターサイズ (タンパク質数)")
    ax1.set_ylabel("クラスター数")
    ax1.set_title("クラスターサイズ分布")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 2. 変動性分布（選択vs非選択）
    variation_threshold_val = cluster_stats_df["Pattern_Std"].quantile(VARIATION_THRESHOLD)

    ax2.hist(cluster_stats_df[~selected_mask]["Pattern_Std"], bins=20, alpha=0.7,
             label=f"非選択 ({(~selected_mask).sum()})", color="lightgray")
    ax2.hist(cluster_stats_df[selected_mask]["Pattern_Std"], bins=20, alpha=0.8,
             label=f"選択 ({selected_mask.sum()})", color="red")
    ax2.axvline(variation_threshold_val, color="blue", linestyle="--",
                label=f"変動性閾値 ({variation_threshold_val:.3f})")
    ax2.set_xlabel("ステージ間変動性 (標準偏差)")
    ax2.set_ylabel("クラスター数")
    ax2.set_title("変動性分布")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # 3. 代表パターンのライングラフ
    colors = plt.cm.tab10(np.arange(len(representative_clusters)))

    for i, cluster_id in enumerate(representative_clusters):
        pattern = cluster_stats_df[cluster_stats_df["Cluster"] == cluster_id]["Pattern"].iloc[0]
        size = cluster_stats_df[cluster_stats_df["Cluster"] == cluster_id]["Size"].iloc[0]
        ax3.plot(range(len(STAGE_ORDER)), pattern,
                 marker="o", linewidth=2, label=f"C{cluster_id} (n={size})", color=colors[i])

    ax3.set_xticks(range(len(STAGE_ORDER)))
    ax3.set_xticklabels(STAGE_ORDER)
    ax3.set_xlabel("ステージ")
    ax3.set_ylabel("Z-score (Stage-normalized)")
    ax3.set_title("選択された代表パターン")
    ax3.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    ax3.grid(True, alpha=0.3)
    ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)

    # 4. パターン間相関ヒートマップ
    selected_patterns = []
    for cluster_id in representative_clusters:
        pattern = cluster_stats_df[cluster_stats_df["Cluster"] == cluster_id]["Pattern"].iloc[0]
        selected_patterns.append(pattern)

    if len(selected_patterns) > 1:
        corr_matrix = np.corrcoef(selected_patterns)
        im = ax4.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")

        # ラベル設定
        cluster_labels = [f"C{cid}" for cid in representative_clusters]
        ax4.set_xticks(range(len(cluster_labels)))
        ax4.set_yticks(range(len(cluster_labels)))
        ax4.set_xticklabels(cluster_labels, rotation=45)
        ax4.set_yticklabels(cluster_labels)

        # 相関係数の数値表示
        for i in range(len(selected_patterns)):
            for j in range(len(selected_patterns)):
                ax4.text(j, i, f"{corr_matrix[i, j]:.2f}",
                        ha="center", va="center", fontsize=9,
                        color="white" if abs(corr_matrix[i, j]) > 0.5 else "black")

        ax4.set_title("選択パターン間相関")

        # カラーバー追加
        plt.colorbar(im, ax=ax4, shrink=0.8, label="Pearson Correlation")

    # 全体レイアウト調整
    plt.suptitle('深層学習DIA: 代表パターン選択結果', fontsize=16, fontweight='bold')
    plt.tight_layout()

    # 図の保存
    import os
    os.makedirs(FIG_DIR, exist_ok=True)
    plt.savefig(f"{FIG_DIR}/fig_pattern_selection_summary_openms.png", dpi=300, bbox_inches="tight")
    plt.show()

# 選択結果の可視化実行
visualize_pattern_selection_result(cluster_stats_df, representative_clusters)

## 選択パターンの詳細統計

選択されたパターンの特徴を定量的に分析します。

In [ ]:
# 選択パターンの統計分析
selected_data = cluster_stats_df[cluster_stats_df["Cluster"].isin(representative_clusters)].copy()

print("=== 選択パターン統計サマリー ===")
print(f"選択パターン数: {len(selected_data)} / {len(cluster_stats_df)}")
print(f"平均クラスターサイズ: {selected_data['Size'].mean():.1f} ± {selected_data['Size'].std():.1f}")
print(f"平均変動性: {selected_data['Pattern_Std'].mean():.3f} ± {selected_data['Pattern_Std'].std():.3f}")
print(f"平均変動幅: {selected_data['Pattern_Range'].mean():.3f} ± {selected_data['Pattern_Range'].std():.3f}")

# パターン間相関の統計
selected_patterns = [row["Pattern"] for _, row in selected_data.iterrows()]
if len(selected_patterns) > 1:
    corr_matrix = np.corrcoef(selected_patterns)
    # 対角線を除く相関係数を取得
    off_diag_corrs = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
    print(f"\nパターン間相関統計:")
    print(f"  平均相関: {np.mean(np.abs(off_diag_corrs)):.3f} ± {np.std(np.abs(off_diag_corrs)):.3f}")
    print(f"  最大相関: {np.max(np.abs(off_diag_corrs)):.3f}")
    print(f"  最小相関: {np.min(np.abs(off_diag_corrs)):.3f}")

# パターンのタイプ分類
print(f"\n=== パターンタイプ分類 ===")
pattern_types = []
for _, row in selected_data.iterrows():
    normal_val = row["Normal_Value"]
    stage4_val = row["StageIV_Value"]
    change = stage4_val - normal_val
    
    if change > 0.5:
        pattern_type = "強上昇型"
    elif change > 0.1:
        pattern_type = "上昇型"
    elif change > -0.1:
        pattern_type = "安定型"
    elif change > -0.5:
        pattern_type = "下降型"
    else:
        pattern_type = "強下降型"
    
    pattern_types.append(pattern_type)

pattern_type_counts = pd.Series(pattern_types).value_counts()
print(pattern_type_counts.to_string())

# 詳細テーブルの作成
detailed_table = selected_data.copy()
detailed_table['Pattern_Type'] = pattern_types
detailed_table['Change_Direction'] = detailed_table.apply(
    lambda row: "↑" if row["StageIV_Value"] > row["Normal_Value"] else "↓", axis=1)
detailed_table['Change_Magnitude'] = detailed_table.apply(
    lambda row: abs(row["StageIV_Value"] - row["Normal_Value"]), axis=1)

print(f"\n=== 選択パターン詳細テーブル ===")
display_cols = ['Cluster', 'Size', 'Pattern_Std', 'Pattern_Type', 'Change_Direction', 'Change_Magnitude']
print(detailed_table[display_cols].to_string(index=False, float_format='%.3f'))

## 選択結果の保存

代表パターンの情報を次回の詳細可視化解析で使用するために保存します。

In [ ]:
# 代表パターン情報の保存
representative_info = cluster_stats_df[cluster_stats_df["Cluster"].isin(representative_clusters)].copy()
representative_info = representative_info.reset_index(drop=True)
representative_info["Pattern_ID"] = range(1, len(representative_info) + 1)

# パターン情報をCSV保存
representative_info_save = representative_info.drop("Pattern", axis=1)  # リスト列は除外
import os
os.makedirs(TABLE_DIR, exist_ok=True)
representative_info_save.to_csv(f"{TABLE_DIR}/representative_patterns_openms.csv", index=False)

# 詳細なパターンデータをNumPy形式で保存
pattern_matrix = np.array([pattern for pattern in representative_info["Pattern"]])
np.save(f"{TABLE_DIR}/representative_pattern_matrix_openms.npy", pattern_matrix)

# 選択されたクラスターIDリストも保存
with open(f"{TABLE_DIR}/representative_cluster_ids_openms.txt", "w") as f:
    f.write(",".join(map(str, representative_clusters)))

print(f"代表パターン保存完了:")
print(f"- パターン情報: {TABLE_DIR}/representative_patterns_openms.csv")
print(f"- パターン行列: {TABLE_DIR}/representative_pattern_matrix_openms.npy")
print(f"- クラスターIDリスト: {TABLE_DIR}/representative_cluster_ids_openms.txt")
print(f"- 選択されたクラスター: {representative_clusters}")

# 保存した情報の確認
print(f"\n=== 保存データ確認 ===")
saved_df = pd.read_csv(f"{TABLE_DIR}/representative_patterns_openms.csv")
saved_matrix = np.load(f"{TABLE_DIR}/representative_pattern_matrix_openms.npy")
print(f"CSV: {saved_df.shape[0]} パターン, {saved_df.shape[1]} 列")
print(f"NumPy: {saved_matrix.shape[0]} パターン × {saved_matrix.shape[1]} ステージ")

## まとめ

50クラスターから**代表的な8パターン**の系統的選択が完了しました：

### パターン選択成果

1. **多基準評価**: サイズ・変動性・多様性を統合した客観的選択
2. **統計的信頼性**: 全選択パターンが10+タンパク質で構成
3. **生物学的意義**: 上位70%の変動性で明瞭なステージ応答を確保
4. **包括性**: 低相関パターンセットで大腸がん進行の多様性を網羅

### 選択基準の効果

- **サイズフィルタ**: 統計的に信頼性の高いクラスターのみを選択
- **変動性フィルタ**: 生物学的意義のある明瞭な変動パターンを確保
- **多様性選択**: 相関の低い独立したパターンセットを構築

これらの8パターンにより、大腸がんステージ進行の分子動態を包括的かつ簡潔に表現する基盤が整いました。

**次のNotebook**: `notebook_15d_openms_stage_visualization.ipynb` — 8パターンの詳細可視化とバイオマーカー候補抽出